# Manual agent benchmark

Compare a **trained agent** loaded from a checkpoint against any subset of the heuristic baselines on a fixed evaluation scenario. Every agent is run on the **same factory layout** for `N_EVAL_EPISODES` rollouts of `MAX_EVAL_SIM_TIME` sim time each.

**What to edit:** only the *User configuration* cell below — the rest is wired up automatically.

**Outputs:**
- Per-episode KPI table (mean & std per agent)
- Bar charts comparing the headline KPIs
- Per-step trend plots (rolling-mean of `repair_time_delta`, `repair_quality`, fleet knowledge) averaged across episodes
- A CSV dump under `reports/benchmark_<timestamp>/`

## 1. User configuration

Point the three paths at the artefacts you want to evaluate, pick which baselines to compare against, and set the horizon. Paths are **relative to the project root** — the Setup cell (section 2) chdirs there before any file is read, so these stay correct regardless of where you launched Jupyter from.

In [ ]:
from pathlib import Path

# --- trained agent ---------------------------------------------------------
# Matches the 3-episode SetTransformer verification run:
#   kata-experiment \
#     --env run_configs/factory_set_quick.json \
#     --agent run_configs/agents/set_transformer.json \
#     --experiment run_configs/experiments/verify_checkpoint.json
TRAINED_ENV_CONFIG   = Path("run_configs/factory_set_quick.json")        # env the agent was trained on
TRAINED_AGENT_CONFIG = Path("run_configs/agents/set_transformer.json")   # matching agent JSON
TRAINED_CHECKPOINT   = Path("checkpoints/set_transformer_final.pt")      # weights to load
TRAINED_AGENT_LABEL  = "SetTransformer (verify)"                         # label used in plots

# --- baselines to compare against ------------------------------------------
# Any subset of: 'random', 'round_robin', 'least_busy', 'least_fatigued', 'shortest_queue'
BASELINES: list[str] = [
    "random",
    "round_robin",
    "least_busy",
    "least_fatigued",
    "shortest_queue",
]

# --- evaluation horizon ----------------------------------------------------
# Defaults mirror factory_set_quick.json's training horizon so the benchmark
# is directly comparable to the training-time eval curve.  Bump
# MAX_EVAL_SIM_TIME above 30_000 to probe extrapolation past the training horizon.
N_EVAL_EPISODES    = 3         # rollouts per agent (more = lower variance, longer wall time)
MAX_EVAL_SIM_TIME  = 30_000.0  # same as factory_set_quick.json's max_sim_time
MAX_EVAL_STEPS     = 3_000     # same as factory_set_quick.json's max_episode_steps
EVAL_SEED          = 4321      # seed for the fixed eval factory (RandomScenarioSampler)
DETERMINISTIC      = True      # use greedy policy for the trained agent

# --- output ----------------------------------------------------------------
REPORTS_ROOT = Path("reports")

print(f"Trained env config : {TRAINED_ENV_CONFIG}")
print(f"Trained agent cfg  : {TRAINED_AGENT_CONFIG}")
print(f"Checkpoint         : {TRAINED_CHECKPOINT}")
print(f"Baselines          : {BASELINES}")
print(f"Episodes / agent   : {N_EVAL_EPISODES}")
print(f"Max sim time       : {MAX_EVAL_SIM_TIME}")
print(f"Max env steps      : {MAX_EVAL_STEPS}")


## 2. Setup

In [ ]:
import json
import logging
import os
import sys
import time
import warnings
from contextlib import contextmanager

# Anchor to the repo root so the relative paths in the user-config cell
# resolve identically whether you launched Jupyter from the project
# root or from examples/.
_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "run_configs").is_dir():
    _root = _root.parent
if (_root / "run_configs").is_dir():
    os.chdir(_root)
    print(f"Working directory: {_root}")
else:
    print(f"WARNING: could not locate repo root from {Path.cwd()}")

# Silence SimPy prints and excess library logging
logging.disable(logging.WARNING)
warnings.filterwarnings("ignore")

# Avoid loading an unrelated default config file
os.environ["KATA_CONF_PATH"] = "/dev/null/__no_file__"

@contextmanager
def quiet():
    """Suppress stdout from the simulation (machine/router/source prints)."""
    old = sys.stdout
    sys.stdout = open(os.devnull, "w")
    try:
        yield
    finally:
        sys.stdout.close()
        sys.stdout = old

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kata.core.config import KATAConfig
from kata.core.tokenizer import StateTokenizer
from kata.env import KataEnv
from kata.scenario import ScenarioBuilder
from kata.EntityFactories import RandomScenarioSampler
from experiment.config import AgentConfig
from agents import (
    LeastBusyAgent,
    LeastFatiguedAgent,
    PPOLatentAgent,
    PPOTransformerAgent,
    RainbowDQNAgent,
    RandomAgent,
    RoundRobinAgent,
    SetTransformerAgent,
    ShortestQueueAgent,
)
from agents.grpo.grpo import GRPOAgent

_AGENT_REGISTRY = {
    "random":          RandomAgent,
    "round_robin":     RoundRobinAgent,
    "least_busy":      LeastBusyAgent,
    "least_fatigued":  LeastFatiguedAgent,
    "shortest_queue":  ShortestQueueAgent,
    "rainbow_dqn":     RainbowDQNAgent,
    "grpo":            GRPOAgent,
    "ppo_transformer": PPOTransformerAgent,
    "ppo_latent":      PPOLatentAgent,
    "set_transformer": SetTransformerAgent,
}
# Token-vocab-aware agents (vocab_size injected at build time).
_TOKEN_AGENTS = {
    "rainbow_dqn", "grpo", "ppo_transformer", "ppo_latent", "set_transformer",
}
# Agents that consume the grouped ``set`` observation rather than the
# flat token stream — their n_actions is the *padded* max_techs cap and
# the encoder is sized by per-slot caps, not a flat seq length.
_SET_OBS_AGENTS = {"set_transformer"}

print("Imports OK")


## 3. Build the shared evaluation scenario

We sample **one** factory from the trained agent's config pool and reuse it for every agent. The horizon is overridden so the benchmark probes long-run behaviour irrespective of the training-time `max_sim_time`. A tokenizer is built from the full type pool so the trained agent never sees unknown tokens.

In [10]:
env_cfg = KATAConfig(**json.loads(TRAINED_ENV_CONFIG.read_text()))

# Stretch the horizon for the benchmark (does not touch the saved env config file)
env_cfg.gym = env_cfg.gym.model_copy(update={
    "max_episode_steps": MAX_EVAL_STEPS,
    "max_sim_time":      MAX_EVAL_SIM_TIME,
})

rcfg = env_cfg.randomized_scenario
if rcfg.enabled:
    sampler = RandomScenarioSampler(env_cfg, rcfg, seed=EVAL_SEED)
    fixed_eval_cfg = sampler.sample_config()
    def scenario_factory(c=fixed_eval_cfg):
        return ScenarioBuilder(c).build()
    n_techs        = rcfg.n_technicians
    machine_types  = sampler.all_machine_types()
    component_types = sampler.all_component_types()
else:
    def scenario_factory(c=env_cfg):
        return ScenarioBuilder(c).build()
    n_techs        = len(env_cfg.technicians)
    machine_types  = sorted({m.machine_type for m in env_cfg.machines.values()})
    component_types = sorted({
        c.component_type
        for m in env_cfg.machines.values()
        for c in m.components.values()
    })

tokenizer = StateTokenizer.build_vocab(
    machine_types=machine_types,
    n_technicians=n_techs,
    seq_length=env_cfg.gym.tokenizer_seq_length,
    component_types=component_types,
    next_ticket_lookahead=env_cfg.gym.next_ticket_lookahead,
)

print(f"Action space size : {n_techs}")
print(f"Machine types     : {machine_types}")
print(f"Tokenizer vocab   : {tokenizer.vocab_size}")

ValidationError: 3 validation errors for KATAConfig
sim.disruptions.dis_dict.injury.trigger
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='random', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing
sim.disruptions.dis_dict.exhaustion.trigger
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='fatigue', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing
sim.disruptions.dis_dict.vacation.trigger
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='periodic', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing

## 4. Instantiate the trained agent and the baselines

The trained agent's `params` are read straight from its training JSON, so the network shape matches the checkpoint. We then `load()` the weights.

Each agent gets its **own** `KataEnv` paired with the same shared `scenario_factory`. Token-based agents (the trained one) use `observation_representation="token_ids"`; heuristic baselines use `"structured"` so they have access to `technician_busy` / `technician_fatigue`.

In [ ]:
def _technician_templates_for_set(env_cfg, sampler_obj=None) -> list[str]:
    """Return the technician-template pool for set-mode vocab building."""
    rcfg = env_cfg.randomized_scenario
    if rcfg.enabled and rcfg.technician_templates:
        return list(rcfg.technician_templates)
    return sorted({
        t.template
        for t in env_cfg.technicians.values()
        if getattr(t, "template", None) is not None
    })


def _build_set_tokenizer(env_cfg, machine_types, component_types):
    """Deterministic enumeration from env-config pools (third fallback)."""
    return StateTokenizer.build_set_vocab(
        machine_types=machine_types,
        component_types=component_types,
        technician_templates=_technician_templates_for_set(env_cfg),
        seq_length=env_cfg.gym.tokenizer_seq_length,
    )


def _load_set_tokenizer(checkpoint_path: Path, env_cfg, machine_types, component_types):
    """Resolve the eval-time set tokenizer.

    Load order (embedded first, because the checkpoint defines what
    its embedding table actually means):
      1. Vocab embedded in the checkpoint (saved by the patched runner
         via :meth:`SetTransformerAgent.attach_vocab`).  Authoritative
         match for whatever the agent was trained against.
      2. Canonical vocab JSON pointed to by ``env_cfg.gym.set_vocab_path``
         — used when the checkpoint pre-dates vocab embedding.
      3. Deterministic rebuild from the env config's template pools —
         last resort.
    """
    # 1. Embedded in checkpoint
    embedded = SetTransformerAgent.peek_vocab(checkpoint_path)
    if embedded:
        tok = StateTokenizer(seq_length=env_cfg.gym.tokenizer_seq_length)
        tok.load_vocab(embedded)
        tok.freeze()
        print(f"  set tokenizer: {tok.vocab_size} tokens (embedded in checkpoint)")
        return tok

    # 2. Canonical artefact
    vocab_path_str = getattr(env_cfg.gym, "set_vocab_path", None)
    if vocab_path_str:
        vocab_path = Path(vocab_path_str)
        if vocab_path.is_file():
            tok = StateTokenizer.from_json(
                vocab_path, seq_length=env_cfg.gym.tokenizer_seq_length
            )
            print(f"  set tokenizer: {tok.vocab_size} tokens "
                  f"(canonical from {vocab_path})")
            return tok

    # 3. Rebuild from env config
    tok = _build_set_tokenizer(env_cfg, machine_types, component_types)
    print(f"  set tokenizer: {tok.vocab_size} tokens (rebuilt from env config)")
    return tok


def _make_env(representation: str, *, shared_tokenizer=None) -> KataEnv:
    """Build a fresh env on the shared scenario_factory, with the given obs format."""
    gym_cfg = env_cfg.gym.model_copy(update={"observation_representation": representation})
    if shared_tokenizer is None:
        shared_tokenizer = tokenizer if representation == "token_ids" else None
    with quiet():
        env = KataEnv(
            scenario_factory=scenario_factory,
            config=gym_cfg,
            tokenizer=shared_tokenizer,
        )
    return env


def _peek_checkpoint_vocab_size(path: Path) -> int | None:
    """Read the embedding-table row count from the checkpoint state dict."""
    import torch
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state = ckpt.get("net") if isinstance(ckpt, dict) else None
    if not isinstance(state, dict):
        state = ckpt if isinstance(ckpt, dict) else {}
    for k, v in state.items():
        if "token_embedding" in k and hasattr(v, "shape") and len(v.shape) == 2:
            return int(v.shape[0])
    return None


agents: dict[str, tuple["Agent", KataEnv]] = {}

# -- trained agent ----------------------------------------------------------
agent_cfg = AgentConfig(**json.loads(TRAINED_AGENT_CONFIG.read_text()))
trained_cls = _AGENT_REGISTRY[agent_cfg.agent_type]
trained_params = dict(agent_cfg.params)

# Pick the env's observation representation for the trained agent.
if agent_cfg.agent_type in _SET_OBS_AGENTS:
    trained_repr = "set"
elif agent_cfg.agent_type in _TOKEN_AGENTS:
    trained_repr = "token_ids"
else:
    trained_repr = "structured"

# Build the right tokenizer for the trained agent's mode.
if agent_cfg.agent_type in _SET_OBS_AGENTS:
    eval_tokenizer = _load_set_tokenizer(
        TRAINED_CHECKPOINT, env_cfg, machine_types, component_types
    )
elif agent_cfg.agent_type in _TOKEN_AGENTS:
    eval_tokenizer = tokenizer
else:
    eval_tokenizer = None

trained_env = _make_env(trained_repr, shared_tokenizer=eval_tokenizer)

# n_actions handshake: set-mode agents use the padded max_techs cap.
if agent_cfg.agent_type in _SET_OBS_AGENTS:
    trained_params["n_actions"] = int(env_cfg.gym.max_techs)
    trained_params.setdefault("max_techs", int(env_cfg.gym.max_techs))
    trained_params.setdefault("max_machines", int(env_cfg.gym.max_machines))
    trained_params.setdefault("env_length", int(env_cfg.gym.set_env_length))
    trained_params.setdefault("sim_time_scale", float(env_cfg.gym.max_sim_time))
else:
    trained_params["n_actions"] = n_techs

# Vocab handshake for token-aware agents.
if agent_cfg.agent_type in _TOKEN_AGENTS:
    ckpt_vocab = _peek_checkpoint_vocab_size(TRAINED_CHECKPOINT)
    if ckpt_vocab is not None and eval_tokenizer is not None and ckpt_vocab != eval_tokenizer.vocab_size:
        raise RuntimeError(
            f"Vocab size mismatch — the checkpoint was trained with {ckpt_vocab} "
            f"tokens but the resolved eval-time tokenizer has "
            f"{eval_tokenizer.vocab_size}.\n"
            f"For set-mode agents this usually means the canonical vocab grew "
            f"after the checkpoint was trained AND the checkpoint has no "
            f"embedded vocab to fall back to.  Solutions: (a) regenerate the "
            f"canonical with --rebuild from the commit that trained this "
            f"checkpoint, (b) retrain against the current canonical, or "
            f"(c) re-save the checkpoint after attach_vocab() with the "
            f"correct vocab attached."
        )
    if eval_tokenizer is not None:
        trained_params.setdefault("vocab_size", eval_tokenizer.vocab_size)
    if agent_cfg.agent_type not in _SET_OBS_AGENTS:
        trained_params.setdefault("max_seq_len", env_cfg.gym.tokenizer_seq_length)

trained_agent = trained_cls(**trained_params)
trained_agent.load(TRAINED_CHECKPOINT)
agents[TRAINED_AGENT_LABEL] = (trained_agent, trained_env)
print(f"Loaded {agent_cfg.agent_type:>16s} ({TRAINED_AGENT_LABEL}) from {TRAINED_CHECKPOINT}")

# -- baselines --------------------------------------------------------------
for name in BASELINES:
    cls = _AGENT_REGISTRY[name]
    env = _make_env("structured")
    agents[name] = (cls(n_actions=n_techs), env)
    print(f"Built  {name:>16s}")

print(f"\nAgents under benchmark: {list(agents.keys())}")


## 5. Run the benchmark

Each `(agent, env)` pair runs `N_EVAL_EPISODES` rollouts. We collect:

- **Episode KPIs** (from the env's terminal `info["metrics"]`): finished products, MTTR, fleet availability, total breakdowns, total assignments…
- **Per-step series** (from `info["metrics"]`): `repair_time_delta_per`, `repair_quality`, `tech_knowledge`. Stored as lists, one per episode.
- **Episode-cumulative reward**.

In [ ]:
EPISODE_KPI_KEYS = [
    "finished_products",
    "mttr",
    "fleet_availability_rate",
    "throughput_rate",
    "total_breakdowns",
    "total_assignments",
    "total_repairs",
    "ill_technician_count",
]
STEP_SERIES_KEYS = ["repair_time_delta_per", "repair_quality"]

def _scalar_step_metrics(metrics: dict) -> dict[str, float]:
    """Drop grouped (`name/subkey`) entries and keep plain scalars."""
    return {k: float(v) for k, v in metrics.items() if "/" not in k}

def _mean_grouped(metrics: dict, prefix: str) -> float | None:
    """Mean across `prefix/<subkey>` entries (e.g. fleet-wide tech_knowledge)."""
    vals = [float(v) for k, v in metrics.items() if k.startswith(prefix + "/")]
    return float(np.mean(vals)) if vals else None

def run_episode(agent, env, *, seed: int) -> dict:
    """Run one rollout to termination and collect metrics."""
    agent.on_episode_start()
    with quiet():
        obs, _info = env.reset(seed=seed)
    ep_reward = 0.0
    step_series: dict[str, list[float]] = {k: [] for k in STEP_SERIES_KEYS}
    fleet_knowledge_series: list[float] = []
    fleet_fatigue_series: list[float] = []
    final_info: dict = {}
    n_steps = 0
    while True:
        action = agent.select_action(obs, deterministic=DETERMINISTIC)
        with quiet():
            obs, reward, terminated, truncated, info = env.step(action)
        ep_reward += float(reward)
        n_steps += 1
        metrics = info.get("metrics", {})
        for k in STEP_SERIES_KEYS:
            if k in metrics:
                step_series[k].append(float(metrics[k]))
        fk = _mean_grouped(metrics, "tech_knowledge")
        if fk is not None:
            fleet_knowledge_series.append(fk)
        ff = _mean_grouped(metrics, "tech_fatigue")
        if ff is not None:
            fleet_fatigue_series.append(ff)
        if terminated or truncated:
            final_info = info
            break
    agent.on_episode_end(ep_reward)
    ep_kpis = _scalar_step_metrics(final_info.get("metrics", {}))
    ep_kpis["episode_reward"] = ep_reward
    ep_kpis["n_steps"]        = n_steps
    ep_kpis["sim_time"]       = float(final_info.get("sim_time", 0.0))
    return {
        "kpis": ep_kpis,
        "step_series": step_series,
        "fleet_knowledge": fleet_knowledge_series,
        "fleet_fatigue": fleet_fatigue_series,
    }

results: dict[str, list[dict]] = {label: [] for label in agents}
t0 = time.time()
for label, (agent, env) in agents.items():
    print(f"--- {label} ---")
    for ep in range(N_EVAL_EPISODES):
        seed = EVAL_SEED * 100 + ep   # deterministic per-episode seed shared across agents
        ep_result = run_episode(agent, env, seed=seed)
        results[label].append(ep_result)
        kp = ep_result["kpis"]
        print(
            f"  ep{ep}  finished={kp.get('finished_products', float('nan')):.0f}"
            f"  mttr={kp.get('mttr', float('nan')):.2f}"
            f"  avail={kp.get('fleet_availability_rate', float('nan')):.3f}"
            f"  return={kp['episode_reward']:+.2f}"
            f"  steps={kp['n_steps']}"
        )
print(f"\nTotal wall time: {time.time() - t0:.1f}s")

## 6. Per-episode KPI summary

Mean ± std across `N_EVAL_EPISODES` episodes for each agent.

In [ ]:
rows = []
for label, ep_list in results.items():
    for ep_idx, ep in enumerate(ep_list):
        row = {"agent": label, "episode": ep_idx}
        row.update(ep["kpis"])
        rows.append(row)
ep_df = pd.DataFrame(rows)

summary_cols = [c for c in (EPISODE_KPI_KEYS + ["episode_reward"]) if c in ep_df.columns]
summary = ep_df.groupby("agent")[summary_cols].agg(["mean", "std"]).round(3)
summary

## 7. Headline KPI bar charts

One panel per KPI. Bars are episode means; error bars are ±1 std.

In [ ]:
plot_kpis = [c for c in [
    "episode_reward",
    "finished_products",
    "fleet_availability_rate",
    "mttr",
    "total_breakdowns",
    "throughput_rate",
] if c in ep_df.columns]

n = len(plot_kpis)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.2 * nrows))
axes = np.atleast_2d(axes).flatten()

agent_order = list(agents.keys())
for i, kpi in enumerate(plot_kpis):
    ax = axes[i]
    means = [ep_df[ep_df.agent == a][kpi].mean() for a in agent_order]
    stds  = [ep_df[ep_df.agent == a][kpi].std(ddof=0) for a in agent_order]
    colors = ["tab:red" if a == TRAINED_AGENT_LABEL else "tab:blue" for a in agent_order]
    ax.bar(agent_order, means, yerr=stds, color=colors, edgecolor="black", linewidth=0.5, capsize=4)
    ax.set_title(kpi)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, axis="y", alpha=0.3)
for j in range(n, len(axes)):
    axes[j].axis("off")
fig.suptitle(f"Agent benchmark ({N_EVAL_EPISODES} eps × max_sim_time={MAX_EVAL_SIM_TIME:.0f})", y=1.02)
fig.tight_layout()
plt.show()

## 8. Per-step trend comparison

Within-episode time-series, averaged across the `N_EVAL_EPISODES` rollouts and smoothed with a rolling mean. Helps see how policies behave *during* a long episode (e.g. knowledge growth, repair-time savings drifting).

In [ ]:
def _stack_series(series_per_episode: list[list[float]]) -> np.ndarray:
    """Pad ragged episodes with NaN to a 2-D array of shape (n_eps, max_len)."""
    if not series_per_episode:
        return np.zeros((0, 0))
    max_len = max(len(s) for s in series_per_episode)
    out = np.full((len(series_per_episode), max_len), np.nan)
    for i, s in enumerate(series_per_episode):
        out[i, :len(s)] = s
    return out

def _rolling_mean(x: np.ndarray, w: int) -> np.ndarray:
    w = max(1, min(int(w), len(x)))
    if w <= 1:
        return x
    out = np.empty_like(x, dtype=float)
    csum = np.concatenate([[0.0], np.nancumsum(x)])
    counts = np.concatenate([[0], np.cumsum(~np.isnan(x))])
    for i in range(len(x)):
        lo = max(0, i + 1 - w)
        denom = max(1, counts[i + 1] - counts[lo])
        out[i] = (csum[i + 1] - csum[lo]) / denom
    return out

# Four trend panels: the two step-series scalars + fleet knowledge + fleet fatigue.
trend_metrics = STEP_SERIES_KEYS + ["fleet_knowledge", "fleet_fatigue"]
fig, axes = plt.subplots(len(trend_metrics), 1, figsize=(9, 3.2 * len(trend_metrics)))
axes = np.atleast_1d(axes)

for ax, metric in zip(axes, trend_metrics):
    for label in agent_order:
        if metric == "fleet_knowledge":
            per_ep = [ep["fleet_knowledge"] for ep in results[label]]
        elif metric == "fleet_fatigue":
            per_ep = [ep["fleet_fatigue"] for ep in results[label]]
        else:
            per_ep = [ep["step_series"].get(metric, []) for ep in results[label]]
        stacked = _stack_series(per_ep)
        if stacked.size == 0:
            continue
        mean_curve = np.nanmean(stacked, axis=0)
        w = max(5, len(mean_curve) // 30)
        smoothed = _rolling_mean(mean_curve, w)
        is_trained = (label == TRAINED_AGENT_LABEL)
        ax.plot(
            range(len(smoothed)),
            smoothed,
            linewidth=2.0 if is_trained else 1.0,
            alpha=1.0 if is_trained else 0.7,
            color="red" if is_trained else None,
            label=label,
        )
    ax.set_title(metric)
    ax.set_xlabel("step within episode")
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.3)
    if metric == "fleet_fatigue":
        ax.set_ylim(0.0, 1.0)  # fatigue is bounded [0, 1]
    ax.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()

## 9. Persist the results

Drops the per-episode KPI table to `reports/benchmark_<timestamp>/episode_metrics.csv` and the per-step series of each agent's first episode (most useful for plot reproduction).

In [ ]:
ts = time.strftime("%Y%m%d-%H%M%S")
out_dir = REPORTS_ROOT / f"benchmark_{ts}"
out_dir.mkdir(parents=True, exist_ok=True)

ep_df.to_csv(out_dir / "episode_metrics.csv", index=False)
summary.to_csv(out_dir / "kpi_summary.csv")

for label, ep_list in results.items():
    if not ep_list:
        continue
    safe_label = label.replace("/", "_").replace(" ", "_")
    first = ep_list[0]
    rows_step = []
    max_len = max(
        [len(s) for s in first["step_series"].values()]
        + [len(first["fleet_knowledge"])]
        + [0]
    )
    for i in range(max_len):
        row = {"step": i}
        for k, series in first["step_series"].items():
            row[k] = series[i] if i < len(series) else None
        row["fleet_knowledge"] = (
            first["fleet_knowledge"][i] if i < len(first["fleet_knowledge"]) else None
        )
        rows_step.append(row)
    pd.DataFrame(rows_step).to_csv(out_dir / f"step_series__{safe_label}.csv", index=False)

manifest = {
    "trained_env_config":   str(TRAINED_ENV_CONFIG),
    "trained_agent_config": str(TRAINED_AGENT_CONFIG),
    "trained_checkpoint":   str(TRAINED_CHECKPOINT),
    "trained_agent_label":  TRAINED_AGENT_LABEL,
    "baselines":            BASELINES,
    "n_eval_episodes":      N_EVAL_EPISODES,
    "max_eval_sim_time":    MAX_EVAL_SIM_TIME,
    "max_eval_steps":       MAX_EVAL_STEPS,
    "eval_seed":            EVAL_SEED,
    "deterministic":        DETERMINISTIC,
    "n_techs":              n_techs,
    "machine_types":        machine_types,
    "component_types":      component_types,
    "tokenizer_vocab_size": tokenizer.vocab_size,
}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))

print(f"Saved benchmark artefacts to {out_dir}")
for p in sorted(out_dir.iterdir()):
    print("  ", p.name)

# ─────────────────────────────────────────────────────────────────────
# PART 2 — Long single-episode benchmark
# ─────────────────────────────────────────────────────────────────────

Part 1 ran *several short rollouts* per agent and aggregated their KPIs.  That's the right protocol for variance-aware comparisons, but it smooths over the *intra-episode* dynamics that the human-centric thesis depends on — knowledge has to actually accumulate, fatigue has to actually swing, and the throughput vs. workforce-investment trade-off only resolves over long horizons.

Part 2 runs **one single really long episode per agent** on the same eval factory.  Useful for:

- Watching the per-step trend curves drift over a much longer window than Part 1's `MAX_EVAL_SIM_TIME` allows.
- Seeing **cumulative finished products** diverge between throughput-optimised and human-centric policies — early lead vs. eventual lead.
- Spotting late-episode failure modes (fatigue lock-in, technician saturation, knowledge plateaus) that short episodes hide.

The same `agents` dict (and underlying `scenario_factory`) is reused, so the long-episode comparison sits on the same factory layout as Part 1's bar charts.

## 10. User configuration for the long episode

Increase the horizon by an order of magnitude over Part 1.  Defaults: `LONG_MAX_SIM_TIME = 2_000_000` (10× the training horizon) and `LONG_MAX_STEPS = 200_000` (also 10×).  A separate `LONG_SEED` keeps the long-run scenario distinct from the short-run one so the two halves of the notebook don't share evaluation noise.

In [ ]:
# Long-episode horizon — 10× the Part-1 defaults.  Heads-up: a single
# episode at this length can take several minutes per agent depending
# on policy speed (the RL agent is slowest because of the per-step
# Transformer forward pass).
LONG_MAX_SIM_TIME = 2_000_000.0
LONG_MAX_STEPS    = 200_000
LONG_SEED         = EVAL_SEED + 7   # distinct from Part 1's seed

print(f"Long-episode horizon : sim_time={LONG_MAX_SIM_TIME:.0f}  steps={LONG_MAX_STEPS}")
print(f"Long-episode seed    : {LONG_SEED}")

## 11. Run one long episode per agent

Each agent gets a freshly-built env (same shared `scenario_factory` as Part 1, but configured for the new horizon) so the long episode is isolated from the per-agent Part-1 buffers.  We track the same per-step series as Part 1 plus a **cumulative finished-products** trace — the canonical visualisation of "who actually produced more output over time".

In [ ]:
def _make_long_env(representation: str, *, shared_tokenizer=None) -> KataEnv:
    """Build a fresh env on the shared scenario_factory at the long horizon.

    For ``token_ids`` we reuse the shared ``tokenizer``.  For ``set``
    we reuse the trained env's already-warmed-up tokenizer so the long
    env doesn't grow new tokens that the agent's embedding hasn't seen.
    """
    long_gym_cfg = env_cfg.gym.model_copy(update={
        "observation_representation": representation,
        "max_sim_time": LONG_MAX_SIM_TIME,
        "max_episode_steps": LONG_MAX_STEPS,
    })
    if shared_tokenizer is None:
        shared_tokenizer = tokenizer if representation == "token_ids" else None
    with quiet():
        env = KataEnv(
            scenario_factory=scenario_factory,
            config=long_gym_cfg,
            tokenizer=shared_tokenizer,
        )
    return env


def _read_total_finished(env) -> int:
    """Sum products absorbed across all sinks of the running env."""
    sinks = getattr(env.dispatcher, "sinks", []) or []
    return sum(int(getattr(s, "completed", 0)) for s in sinks)


def run_long_episode(agent, env, *, seed: int) -> dict:
    """One single long rollout; collects per-step traces + cumulative throughput."""
    agent.on_episode_start()
    with quiet():
        obs, _info = env.reset(seed=seed)
    ep_reward = 0.0
    step_series: dict[str, list[float]] = {k: [] for k in STEP_SERIES_KEYS}
    fleet_knowledge_series: list[float] = []
    fleet_fatigue_series: list[float] = []
    cumulative_finished: list[int] = []
    sim_times: list[float] = []
    final_info: dict = {}
    n_steps = 0
    while True:
        action = agent.select_action(obs, deterministic=DETERMINISTIC)
        with quiet():
            obs, reward, terminated, truncated, info = env.step(action)
        ep_reward += float(reward)
        n_steps += 1
        metrics = info.get("metrics", {})
        for k in STEP_SERIES_KEYS:
            if k in metrics:
                step_series[k].append(float(metrics[k]))
        fk = _mean_grouped(metrics, "tech_knowledge")
        if fk is not None:
            fleet_knowledge_series.append(fk)
        ff = _mean_grouped(metrics, "tech_fatigue")
        if ff is not None:
            fleet_fatigue_series.append(ff)
        cumulative_finished.append(_read_total_finished(env))
        sim_times.append(float(info.get("sim_time", 0.0)))
        if terminated or truncated:
            final_info = info
            break
    agent.on_episode_end(ep_reward)
    ep_kpis = _scalar_step_metrics(final_info.get("metrics", {}))
    ep_kpis["episode_reward"] = ep_reward
    ep_kpis["n_steps"]        = n_steps
    ep_kpis["sim_time"]       = float(final_info.get("sim_time", 0.0))
    return {
        "kpis": ep_kpis,
        "step_series": step_series,
        "fleet_knowledge": fleet_knowledge_series,
        "fleet_fatigue": fleet_fatigue_series,
        "cumulative_finished": cumulative_finished,
        "sim_times": sim_times,
    }


# Rebuild one env per agent at the long horizon; reuse the same agent
# instances from Part 1 (PPO weights, baseline state).
long_envs: dict[str, KataEnv] = {}
for label, (agent_obj, env_obj) in agents.items():
    short_repr = env_obj.config.observation_representation
    if short_repr == "set":
        # Reuse the warmed-up tokenizer from the short env so the long
        # env doesn't grow tokens the agent has no embedding for.
        long_envs[label] = _make_long_env(
            "set", shared_tokenizer=env_obj._tokenizer
        )
    elif short_repr == "token_ids":
        long_envs[label] = _make_long_env("token_ids")
    else:
        long_envs[label] = _make_long_env("structured")

long_results: dict[str, dict] = {}
t0 = time.time()
for label, (agent, _short_env) in agents.items():
    print(f"--- {label} (long episode) ---")
    res = run_long_episode(agent, long_envs[label], seed=LONG_SEED)
    long_results[label] = res
    kp = res["kpis"]
    print(
        f"  sim_time={kp['sim_time']:.0f}"
        f"  steps={kp['n_steps']}"
        f"  finished={kp.get('finished_products', float('nan')):.0f}"
        f"  mttr={kp.get('mttr', float('nan')):.2f}"
        f"  avail={kp.get('fleet_availability_rate', float('nan')):.3f}"
        f"  return={kp['episode_reward']:+.2f}"
    )
print(f"\nPart-2 wall time: {time.time() - t0:.1f}s")


## 12. Within-episode trends (long horizon)

Same four series as Part 1 (`repair_time_delta_per`, `repair_quality`, fleet `tech_knowledge`, fleet `tech_fatigue`), but plotted from **one single long rollout per agent** rather than averaged across short ones.  The longer x-axis lets late-episode dynamics actually appear.

In [ ]:
long_trend_metrics = STEP_SERIES_KEYS + ["fleet_knowledge", "fleet_fatigue"]
fig, axes = plt.subplots(len(long_trend_metrics), 1, figsize=(10, 3.0 * len(long_trend_metrics)))
axes = np.atleast_1d(axes)

for ax, metric in zip(axes, long_trend_metrics):
    for label in agent_order:
        res = long_results[label]
        if metric == "fleet_knowledge":
            series = res["fleet_knowledge"]
        elif metric == "fleet_fatigue":
            series = res["fleet_fatigue"]
        else:
            series = res["step_series"].get(metric, [])
        if not series:
            continue
        arr = np.asarray(series, dtype=float)
        # Rolling-mean smoothing — window ~ 1 % of the episode so the
        # signal is legible without losing the late-episode shape.
        w = max(20, len(arr) // 100)
        smoothed = _rolling_mean(arr, w)
        is_trained = (label == TRAINED_AGENT_LABEL)
        ax.plot(
            range(len(smoothed)),
            smoothed,
            linewidth=2.0 if is_trained else 1.0,
            alpha=1.0 if is_trained else 0.7,
            color="red" if is_trained else None,
            label=label,
        )
    ax.set_title(f"{metric}  (single long episode)")
    ax.set_xlabel("decision step within episode")
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.3)
    if metric == "fleet_fatigue":
        ax.set_ylim(0.0, 1.0)
    ax.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()

## 13. Cumulative finished products

The headline visual for the human-centric thesis: cumulative finished-products vs. simulation time, one curve per agent.  A throughput-optimised baseline tends to take an *early* lead; a human-centric agent that invests in knowledge / fatigue balance typically lags early and overtakes later as the workforce specialises and the baseline's strongest tech burns out.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for label in agent_order:
    res = long_results[label]
    sim_t = np.asarray(res["sim_times"], dtype=float)
    finished = np.asarray(res["cumulative_finished"], dtype=float)
    if sim_t.size == 0:
        continue
    is_trained = (label == TRAINED_AGENT_LABEL)
    ax.plot(
        sim_t, finished,
        linewidth=2.0 if is_trained else 1.0,
        alpha=1.0 if is_trained else 0.7,
        color="red" if is_trained else None,
        label=label,
    )
ax.set_xlabel("simulation time")
ax.set_ylabel("cumulative finished products")
ax.set_title("Cumulative throughput over the long episode")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=9)
fig.tight_layout()
plt.show()

# Sanity readout: how the final counts compare.
final_counts = {label: long_results[label]["cumulative_finished"][-1]
                if long_results[label]["cumulative_finished"] else 0
                for label in agent_order}
print("Final cumulative finished products:")
for label, n in sorted(final_counts.items(), key=lambda kv: -kv[1]):
    marker = " ← trained" if label == TRAINED_AGENT_LABEL else ""
    print(f"  {label:<30s} {n:>8d}{marker}")

## 14. Long-episode KPI table

A single row per agent: the terminal KPI bundle from this one long rollout.  Useful as a compact tabular summary alongside the cumulative-throughput plot.

In [ ]:
long_rows = []
for label in agent_order:
    row = {"agent": label}
    row.update(long_results[label]["kpis"])
    row["cumulative_finished_final"] = (
        long_results[label]["cumulative_finished"][-1]
        if long_results[label]["cumulative_finished"]
        else 0
    )
    long_rows.append(row)
long_ep_df = pd.DataFrame(long_rows).set_index("agent")
display_cols = [c for c in (
    EPISODE_KPI_KEYS + ["episode_reward", "cumulative_finished_final", "n_steps", "sim_time"]
) if c in long_ep_df.columns]
long_ep_df[display_cols].round(3)

## 15. Persist the long-episode results

Saved separately from Part 1 under `reports/benchmark_long_<timestamp>/` so the two halves of the notebook don't overwrite each other.

In [ ]:
ts_long = time.strftime("%Y%m%d-%H%M%S")
long_dir = REPORTS_ROOT / f"benchmark_long_{ts_long}"
long_dir.mkdir(parents=True, exist_ok=True)

long_ep_df.to_csv(long_dir / "long_episode_kpis.csv")

# One CSV per agent containing the full per-step trace + cumulative throughput.
for label, res in long_results.items():
    safe = label.replace("/", "_").replace(" ", "_")
    max_len = max(
        [len(s) for s in res["step_series"].values()]
        + [len(res["fleet_knowledge"]), len(res["fleet_fatigue"]),
           len(res["cumulative_finished"]), len(res["sim_times"])]
        + [0]
    )
    rows_step = []
    for i in range(max_len):
        row = {"step": i}
        for k, series in res["step_series"].items():
            row[k] = series[i] if i < len(series) else None
        row["fleet_knowledge"] = (
            res["fleet_knowledge"][i] if i < len(res["fleet_knowledge"]) else None
        )
        row["fleet_fatigue"] = (
            res["fleet_fatigue"][i] if i < len(res["fleet_fatigue"]) else None
        )
        row["cumulative_finished"] = (
            res["cumulative_finished"][i]
            if i < len(res["cumulative_finished"]) else None
        )
        row["sim_time"] = (
            res["sim_times"][i] if i < len(res["sim_times"]) else None
        )
        rows_step.append(row)
    pd.DataFrame(rows_step).to_csv(long_dir / f"step_series__{safe}.csv", index=False)

manifest_long = {
    "trained_env_config":   str(TRAINED_ENV_CONFIG),
    "trained_agent_config": str(TRAINED_AGENT_CONFIG),
    "trained_checkpoint":   str(TRAINED_CHECKPOINT),
    "trained_agent_label":  TRAINED_AGENT_LABEL,
    "baselines":            BASELINES,
    "long_max_sim_time":    LONG_MAX_SIM_TIME,
    "long_max_steps":       LONG_MAX_STEPS,
    "long_seed":            LONG_SEED,
    "deterministic":        DETERMINISTIC,
    "n_techs":              n_techs,
    "machine_types":        machine_types,
    "component_types":      component_types,
    "tokenizer_vocab_size": tokenizer.vocab_size,
}
(long_dir / "manifest.json").write_text(json.dumps(manifest_long, indent=2))

print(f"Saved long-episode benchmark artefacts to {long_dir}")
for p in sorted(long_dir.iterdir()):
    print("  ", p.name)

# ─────────────────────────────────────────────────────────────────────
# PART 3 — Massive-scale long-episode benchmark
# ─────────────────────────────────────────────────────────────────────

Part 2 ran every agent on the *same* small factory layout the agent was trained on.  Part 3 stress-tests **generalisation across scale** — does the SetTransformer's architectural inductive bias (pointer head over a padded candidate set + cross-attention to machines/env) survive a much larger fleet and a much richer machine pool than it ever saw during training?

The trained agent's `max_techs=30` and `max_machines=100` were set up to *accommodate* this scenario from day one — the padded slots simply weren't used in the small-factory training.  Here we point at `benchmark_suite/massive_scale.json`, which fills all 30 tech slots, 80–120 machines drawn from all 12 templates with components, and uses the same canonical vocab artefact as training.  No re-training: we're asking whether the architecture transfers.

Baselines are also rebuilt on the massive env so the comparison is apples-to-apples.


## 16. User configuration for the massive-scale run

The defaults below cap a single massive-scale episode at `MASSIVE_MAX_SIM_TIME = 500_000` and `MASSIVE_MAX_STEPS = 50_000` — a quarter of the scenario's nominal 2M / 200k horizon, because each decision step is more expensive on a 30-tech × 100-machine fleet.  Bump them when you're ready to spend the wall-time.


In [ ]:
MASSIVE_ENV_CONFIG  = Path("run_configs/benchmark_suite/massive_scale.json")
MASSIVE_MAX_SIM_TIME = 500_000.0   # 1/4 of the scenario's nominal 2M
MASSIVE_MAX_STEPS    = 50_000      # 1/4 of the scenario's nominal 200k
MASSIVE_SEED         = EVAL_SEED + 23

print(f"Massive env config  : {MASSIVE_ENV_CONFIG}")
print(f"Massive horizon     : sim_time={MASSIVE_MAX_SIM_TIME:.0f}  steps={MASSIVE_MAX_STEPS}")
print(f"Massive seed        : {MASSIVE_SEED}")


## 17. Build the massive-scale envs and run one episode per agent

Reuses the trained `set_transformer` weights as-is — only the *env* changes.  Baselines get fresh instances at the new `n_actions` (30 padded tech slots).


In [ ]:
massive_env_cfg = KATAConfig(**json.loads(MASSIVE_ENV_CONFIG.read_text()))

# Override the horizon for this benchmark.
massive_env_cfg.gym = massive_env_cfg.gym.model_copy(update={
    "max_sim_time":      MASSIVE_MAX_SIM_TIME,
    "max_episode_steps": MASSIVE_MAX_STEPS,
})

# Sample a single massive-scale scenario and reuse it across agents.
m_rcfg = massive_env_cfg.randomized_scenario
m_sampler = RandomScenarioSampler(massive_env_cfg, m_rcfg, seed=MASSIVE_SEED)
m_fixed_cfg = m_sampler.sample_config()
def massive_scenario_factory(c=m_fixed_cfg):
    return ScenarioBuilder(c).build()

# Resolve the eval-time set tokenizer for the trained agent using the same
# load order Part 1 used (embedded > canonical > rebuild).
m_machine_types = m_sampler.all_machine_types()
m_component_types = m_sampler.all_component_types()
m_n_techs = m_rcfg.n_technicians
print(f"massive scenario : techs={m_n_techs}  machines={m_rcfg.n_machines_min}..{m_rcfg.n_machines_max}")

# Build per-agent envs on the massive scenario factory.
def _make_massive_env(representation: str, *, shared_tokenizer=None) -> KataEnv:
    gym_cfg = massive_env_cfg.gym.model_copy(update={
        "observation_representation": representation,
    })
    with quiet():
        env = KataEnv(
            scenario_factory=massive_scenario_factory,
            config=gym_cfg,
            tokenizer=shared_tokenizer,
        )
    return env

# Resolve set-mode tokenizer using the SAME helper as Part 1 (embedded > canonical > rebuild).
massive_set_tok = (
    _load_set_tokenizer(TRAINED_CHECKPOINT, massive_env_cfg, m_machine_types, m_component_types)
    if agent_cfg.agent_type in _SET_OBS_AGENTS else None
)

massive_agents: dict[str, tuple["Agent", KataEnv]] = {}

# -- trained agent on the massive env -------------------------------------
if agent_cfg.agent_type in _SET_OBS_AGENTS:
    m_trained_env = _make_massive_env("set", shared_tokenizer=massive_set_tok)
elif agent_cfg.agent_type in _TOKEN_AGENTS:
    m_trained_env = _make_massive_env("token_ids", shared_tokenizer=tokenizer)
else:
    m_trained_env = _make_massive_env("structured")
massive_agents[TRAINED_AGENT_LABEL] = (trained_agent, m_trained_env)

# -- baselines (fresh instances at the massive fleet size) ----------------
for name in BASELINES:
    cls = _AGENT_REGISTRY[name]
    env = _make_massive_env("structured")
    massive_agents[name] = (cls(n_actions=m_n_techs), env)

print(f"massive agents under benchmark: {list(massive_agents.keys())}")

# -- run one long episode per agent ---------------------------------------
massive_results: dict[str, dict] = {}
t0 = time.time()
for label, (agent, env_obj) in massive_agents.items():
    print(f"--- {label} (massive) ---")
    res = run_long_episode(agent, env_obj, seed=MASSIVE_SEED)
    massive_results[label] = res
    kp = res["kpis"]
    print(
        f"  sim_time={kp['sim_time']:.0f}"
        f"  steps={kp['n_steps']}"
        f"  finished={kp.get('finished_products', float('nan')):.0f}"
        f"  mttr={kp.get('mttr', float('nan')):.2f}"
        f"  avail={kp.get('fleet_availability_rate', float('nan')):.3f}"
        f"  return={kp['episode_reward']:+.2f}"
    )
print(f"\nPart-3 wall time: {time.time() - t0:.1f}s")


## 18. Massive-scale plots and KPI table

Same four within-episode trends and cumulative-throughput plot as Part 2, on the massive scenario.  Useful for spotting whether the architecture's per-step quality holds up at 30× the fleet size and ~5× the machine count.


In [ ]:
# Within-episode trends -----------------------------------------------------
fig, axes = plt.subplots(len(long_trend_metrics), 1, figsize=(10, 3.0 * len(long_trend_metrics)))
axes = np.atleast_1d(axes)
for ax, metric in zip(axes, long_trend_metrics):
    for label in massive_agents:
        res = massive_results[label]
        if metric == "fleet_knowledge":
            series = res["fleet_knowledge"]
        elif metric == "fleet_fatigue":
            series = res["fleet_fatigue"]
        else:
            series = res["step_series"].get(metric, [])
        if not series:
            continue
        arr = np.asarray(series, dtype=float)
        w = max(20, len(arr) // 100)
        smoothed = _rolling_mean(arr, w)
        is_trained = (label == TRAINED_AGENT_LABEL)
        ax.plot(range(len(smoothed)), smoothed,
                linewidth=2.0 if is_trained else 1.0,
                alpha=1.0 if is_trained else 0.7,
                color="red" if is_trained else None,
                label=label)
    ax.set_title(f"{metric}  (massive-scale single episode)")
    ax.set_xlabel("decision step within episode")
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.3)
    if metric == "fleet_fatigue":
        ax.set_ylim(0.0, 1.0)
    ax.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()

# Cumulative finished products ---------------------------------------------
fig, ax = plt.subplots(figsize=(10, 4.5))
for label in massive_agents:
    res = massive_results[label]
    sim_t = np.asarray(res["sim_times"], dtype=float)
    finished = np.asarray(res["cumulative_finished"], dtype=float)
    if sim_t.size == 0:
        continue
    is_trained = (label == TRAINED_AGENT_LABEL)
    ax.plot(sim_t, finished,
            linewidth=2.0 if is_trained else 1.0,
            alpha=1.0 if is_trained else 0.7,
            color="red" if is_trained else None,
            label=label)
ax.set_xlabel("simulation time")
ax.set_ylabel("cumulative finished products")
ax.set_title("Cumulative throughput (massive scale)")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=9)
fig.tight_layout()
plt.show()

print("Final cumulative finished products (massive scale):")
final_counts = {label: massive_results[label]["cumulative_finished"][-1]
                if massive_results[label]["cumulative_finished"] else 0
                for label in massive_agents}
for label, n in sorted(final_counts.items(), key=lambda kv: -kv[1]):
    marker = " ← trained" if label == TRAINED_AGENT_LABEL else ""
    print(f"  {label:<30s} {n:>8d}{marker}")

# KPI table -----------------------------------------------------------------
massive_rows = []
for label, res in massive_results.items():
    row = {"agent": label}
    row.update(res["kpis"])
    row["cumulative_finished_final"] = (
        res["cumulative_finished"][-1] if res["cumulative_finished"] else 0
    )
    massive_rows.append(row)
massive_ep_df = pd.DataFrame(massive_rows).set_index("agent")
display_cols = [c for c in (
    EPISODE_KPI_KEYS + ["episode_reward", "cumulative_finished_final", "n_steps", "sim_time"]
) if c in massive_ep_df.columns]
display(massive_ep_df[display_cols].round(3))

# Persist -------------------------------------------------------------------
ts_m = time.strftime("%Y%m%d-%H%M%S")
massive_dir = REPORTS_ROOT / f"benchmark_massive_{ts_m}"
massive_dir.mkdir(parents=True, exist_ok=True)
massive_ep_df.to_csv(massive_dir / "massive_episode_kpis.csv")
for label, res in massive_results.items():
    safe = label.replace("/", "_").replace(" ", "_")
    max_len = max(
        [len(s) for s in res["step_series"].values()] +
        [len(res["fleet_knowledge"]), len(res["fleet_fatigue"]),
         len(res["cumulative_finished"]), len(res["sim_times"])] + [0]
    )
    rows_step = []
    for i in range(max_len):
        row = {"step": i}
        for k, series in res["step_series"].items():
            row[k] = series[i] if i < len(series) else None
        row["fleet_knowledge"] = res["fleet_knowledge"][i] if i < len(res["fleet_knowledge"]) else None
        row["fleet_fatigue"]   = res["fleet_fatigue"][i]   if i < len(res["fleet_fatigue"])   else None
        row["cumulative_finished"] = res["cumulative_finished"][i] if i < len(res["cumulative_finished"]) else None
        row["sim_time"]        = res["sim_times"][i] if i < len(res["sim_times"]) else None
        rows_step.append(row)
    pd.DataFrame(rows_step).to_csv(massive_dir / f"step_series__{safe}.csv", index=False)

(massive_dir / "manifest.json").write_text(json.dumps({
    "massive_env_config":   str(MASSIVE_ENV_CONFIG),
    "trained_agent_config": str(TRAINED_AGENT_CONFIG),
    "trained_checkpoint":   str(TRAINED_CHECKPOINT),
    "trained_agent_label":  TRAINED_AGENT_LABEL,
    "massive_max_sim_time": MASSIVE_MAX_SIM_TIME,
    "massive_max_steps":    MASSIVE_MAX_STEPS,
    "massive_seed":         MASSIVE_SEED,
    "n_techs":              m_n_techs,
    "machine_types":        m_machine_types,
    "component_types":      m_component_types,
    "deterministic":        DETERMINISTIC,
}, indent=2))
print(f"\nSaved massive-scale benchmark artefacts to {massive_dir}")
